In [1]:
import numpy as np
from pathlib import Path
import sys

# Ensure we can import from the local `code` and `julia` directories
sys.path.append(str(Path.cwd() / "code"))
sys.path.append(str(Path.cwd() / "julia"))

from displays import create_video_scenario_burnmap
from dataset import load_scenario, load_burn_map
from benchmark import run_benchmark_scenario, return_no_custom_parameters
import wrappers

Initializing the Julia session. This can take up to 1 minute.


initializing the ground sensor julia module
initializing the drone julia module
initializing the TOP julia module
Julia session initialized.


In [2]:
# === Load Scenario ===
scenario = load_scenario("MiniTractDataset/AugustComplexFire/scenarii/AugustComplexFire_scenario1.npy")
print("Scenario shape:", scenario.shape)

Scenario shape: (12, 2067, 2252)


In [3]:
# === Configuration ===

# Burn map and mask for AugustComplexFire
burnmap_filename = "MiniTractDataset/AugustComplexFire/static_risk_whp.npy"
mask_filename = "MiniTractDataset/AugustComplexFire/mask.npy"

# Simulation parameters (same as experiments.ipynb, but with transmission_range = 2900)
simulation_parameters = {
    "max_battery_distance": -1,
    "max_battery_time": 1,
    "n_drones": 2,
    "n_ground_stations": 8,
    "n_charging_stations": 2,
    "drone_speed_m_per_min": 600,
    "coverage_radius_m": 2900,
    "cell_size_m": 30,
    "transmission_range": 50000,
}

# Custom initialization parameters for the strategies
custom_initialization_parameters = {
    "burnmap_filename": burnmap_filename,
    "mask_filename": mask_filename,
    "load_from_logfile": False,
    "reevaluation_step": 5,
    "optimization_horizon": 10,
    "regularization_param": 1e5,
    "recompute_kernel": False,
}

# === Strategy Selection ===
# Random sensor placement (logged)
SensorPlacementStrategy = wrappers.RandomSensorPlacementStrategyLogged
# TOP drone routing with clustering, logging, and mask support
DroneRoutingStrategy = wrappers.DroneRoutingTOPMaskedLogged

print("Sensor strategy:", SensorPlacementStrategy.__name__)
print("Drone strategy:", DroneRoutingStrategy.__name__)
print("Simulation parameters:", simulation_parameters)

Sensor strategy: RandomSensorPlacementStrategyLogged
Drone strategy: DroneRoutingTOPMasked
Simulation parameters: {'max_battery_distance': -1, 'max_battery_time': 1, 'n_drones': 2, 'n_ground_stations': 8, 'n_charging_stations': 2, 'drone_speed_m_per_min': 600, 'coverage_radius_m': 2900, 'cell_size_m': 30, 'transmission_range': 50000}


In [ ]:
# === Run Benchmark ===
print("Running benchmark...")

results, history = run_benchmark_scenario(
    scenario=scenario,
    sensor_placement_strategy=SensorPlacementStrategy,
    drone_routing_strategy=DroneRoutingStrategy,
    custom_initialization_parameters=custom_initialization_parameters,
    custom_step_parameters_function=return_no_custom_parameters,
    starting_time=0,
    return_history=True,
    return_history_scale='operational',
    input_dir='MiniTractDataset/AugustComplexFire/',
    simulation_parameters=simulation_parameters,
)

print("Starting Benchmark")
drone_locations_history, ground_sensor_locations, charging_stations_locations = history
print(f"\nBenchmark complete. Total timesteps: {len(drone_locations_history)}")

In [ ]:
# === Results ===
print("=" * 60)
print("BENCHMARK RESULTS")
print("=" * 60)
for key, value in results.items():
    print(f"  {key}: {value}")
print()
print(f"Ground sensor locations: {ground_sensor_locations}")
print(f"Charging station locations: {charging_stations_locations}")
print(f"Number of drones: {simulation_parameters['n_drones']}")
print(f"Coverage radius: {simulation_parameters['coverage_radius_m']}m")

In [ ]:
# === Render trajectory video ===
# Use the latest temporary burn map from tmp_burnmaps/
burn_map_video = load_burn_map("./tmp_burnmaps/tmp_burnmap_760122.npy")[:len(drone_locations_history)]

out_name = "benchmark_random_TOP_augustcomplex"
create_video_scenario_burnmap(
    burn_map=burn_map_video,
    drone_locations_history=drone_locations_history,
    out_filename=out_name,
    ground_sensor_locations=ground_sensor_locations,
    charging_stations_locations=charging_stations_locations,
    frames_per_image=3,
    maxframes=np.inf,
    display_zones=True,
)

print(f"Done. Video saved to display_{out_name}/{out_name}.mp4")